# Hierarchical CMF vs Traditional Count Models (General Audience)

This notebook compares hierarchical CMF and traditional count models using the Washington Example 16-3 dataset.

Scope for this version:
- includes baseline and random-parameter models
- excludes latent class analysis
- excludes zero-inflated analysis

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from metacountregressor import (
    load_example16_3_model_data,
    ExperimentBuilder,
    CMFExperimentBuilder,
    ModelConstraints,
    load_book_nb_baseline_spec,
    load_book_cmf_spec,
    describe_book_nb_baseline_spec,
    describe_book_cmf_spec,
    extract_summary,
    extract_search_best,
    compare_models,
)

def compute_metrics(observed, preds):
    residuals = observed - preds
    rmse = np.sqrt(np.mean(residuals ** 2))
    mae = np.mean(np.abs(residuals))
    corr = np.corrcoef(observed, preds)[0, 1] if len(observed) > 1 else 0.0
    return {'RMSE': rmse, 'MAE': mae, 'Correlation': corr}

print('Environment ready.')

In [ ]:
df = load_example16_3_model_data()
print('Shape:', df.shape)
df.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df['FREQ'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Crash Frequency Distribution')
axes[1].hist(df['AADT'], bins=30, color='coral', edgecolor='white')
axes[1].set_title('AADT Distribution')
axes[2].scatter(df['AADT'], df['FREQ'], s=18, alpha=0.5)
axes[2].set_title('FREQ vs AADT')
plt.tight_layout()
plt.show()

mean_val = df['FREQ'].mean()
var_val = df['FREQ'].var()
print(f'Overdispersion ratio (Var/Mean): {var_val/mean_val:.2f}')

## 1) Traditional Count Models

We fit:
- baseline fixed-effects NB
- random-parameter NB

In [ ]:
builder = ExperimentBuilder(df=df, id_col='ID', y_col='FREQ', offset_col='OFFSET')

spec_nb_baseline = load_book_nb_baseline_spec()
describe_book_nb_baseline_spec()

fit_nb_baseline = builder.fit_manual_model(manual_spec=spec_nb_baseline, model='nb', R=200)
sum_nb_baseline = extract_summary(fit_nb_baseline)
print('NB baseline BIC:', round(sum_nb_baseline['bic'], 2))

spec_nb_random = builder.make_manual_spec(
    fixed_terms=['URB', 'ACCESS', 'GRADEBR'],
    rdm_terms=['CURVES:lognormal', 'LENGTH:normal'],
    dispersion=1,
    latent_classes=1,
)
fit_nb_random = builder.fit_manual_model(manual_spec=spec_nb_random, model='nb', R=200)
sum_nb_random = extract_summary(fit_nb_random)
print('NB random BIC:', round(sum_nb_random['bic'], 2))

## 2) Hierarchical CMF Models

We fit:
- baseline CMF
- random-parameter CMF

In [ ]:
cmf_spec = load_book_cmf_spec()
describe_book_cmf_spec()

cmf_builder = CMFExperimentBuilder(
    df=df,
    y_col='FREQ',
    aadt_col='AADT',
    baseline_vars=cmf_spec['baseline_vars'],
    local_vars=cmf_spec['local_vars'],
)

manual_cmf_base = cmf_builder.make_manual_cmf_spec(
    baseline_fixed=['URB', 'ACCESS', 'GRADEBR', 'CURVES'],
    local_fixed=['CURVES', 'WIDTH'],
    dispersion=1,
)
fit_cmf_base = cmf_builder.fit_manual_cmf_model(id_col='ID', manual_spec=manual_cmf_base, model='nb', R=200)
sum_cmf_base = extract_summary(fit_cmf_base)
print('CMF baseline BIC:', round(sum_cmf_base['bic'], 2))

manual_cmf_random = cmf_builder.make_manual_cmf_spec(
    baseline_fixed=['URB', 'ACCESS', 'GRADEBR'],
    baseline_random=['CURVES:lognormal'],
    local_fixed=['CURVES', 'WIDTH'],
    dispersion=1,
)
fit_cmf_random = cmf_builder.fit_manual_cmf_model(id_col='ID', manual_spec=manual_cmf_random, model='nb', R=200)
sum_cmf_random = extract_summary(fit_cmf_random)
print('CMF random BIC:', round(sum_cmf_random['bic'], 2))

## 3) Automated Search (No Latent Class, No Zero Inflation)

Search space in this notebook is intentionally constrained to improve interpretability for a general audience.

In [ ]:
constraints = ModelConstraints().force_include('AADT').force_fixed('AADT')

eval_count = builder.build_count_evaluator(
    variables=['URB', 'ACCESS', 'GRADEBR', 'CURVES', 'LENGTH', 'WIDTH', 'SPEED', 'SLOPE'],
    default_roles=[0, 1, 2],
    max_latent_classes=1,
    R=200,
    constraints=constraints,
)
search_count_raw = builder.run(evaluator=eval_count, algo='sa', max_iter=400, seed=42)
search_count = extract_search_best(search_count_raw)

best_count_spec = eval_count.build_spec(search_count['best_decision'])
fit_best_count = builder.fit_manual_model(manual_spec=best_count_spec, model='nb', R=300)

cmf_builder_search = CMFExperimentBuilder(
    df=df,
    y_col='FREQ',
    aadt_col='AADT',
    baseline_vars=['URB', 'ACCESS', 'GRADEBR', 'CURVES', 'LENGTH'],
    local_vars=['CURVES', 'WIDTH', 'SLOPE'],
)
cmf_eb, eval_cmf, _meta = cmf_builder_search.build_jax_count_evaluator(
    id_col='ID',
    offset_col='OFFSET',
    default_roles=[0, 1, 2],
    max_latent_classes=1,
    R=200,
)
search_cmf_raw = cmf_eb.run(evaluator=eval_cmf, algo='sa', max_iter=400, seed=42)
search_cmf = extract_search_best(search_cmf_raw)

best_cmf_spec = eval_cmf.build_spec(search_cmf['best_decision'])
fit_best_cmf = cmf_eb.fit_manual_model(manual_spec=best_cmf_spec, model='nb', R=300)

print('Best count BIC from search:', round(search_count['best_bic'], 2))
print('Best CMF BIC from search:', round(search_cmf['best_bic'], 2))

In [ ]:
fits = {
    'NB Baseline': fit_nb_baseline,
    'NB Random': fit_nb_random,
    'CMF Baseline': fit_cmf_base,
    'CMF Random': fit_cmf_random,
    'Best Count (Search)': fit_best_count,
    'Best CMF (Search)': fit_best_cmf,
}

comp = compare_models(fits)
comp

In [ ]:
observed = df['FREQ'].values
rows = []
for name, fit in fits.items():
    preds = fit.get('predictions')
    if preds is None:
        continue
    preds = np.asarray(preds).flatten()
    if len(preds) != len(observed):
        continue
    m = compute_metrics(observed, preds)
    rows.append({'Model': name, **m})
val = pd.DataFrame(rows).sort_values('RMSE')
val

## 4) Conclusion for a General Audience

- Traditional NB provides a strong and familiar baseline.
- Hierarchical CMF offers clearer policy interpretation via CMF-style structure.
- Random parameters improve realism in both frameworks.
- Use BIC and validation metrics together when selecting the final model.